# tp-05n 多様なメンバーの追加（Kaggle 上位解法の「多様性」を取り入れる）

Kaggle の金メダル解法の調査（NVIDIA Grandmasters Playbook、TabArena、TabM、各コンペの解法記事）で共通していたのは、**学習原理の違うモデルを多数そろえ、OOF で合成する**こと。現行の 7 モデルに足りない原理を追加する。特徴量は v2 の LightGBM と同じ（追い切り・厩舎コメント・血統×条件を含む 約 250 列）、OOF は valid 2022〜2024 × 3 seed。

| メンバー | 狙い |
|---|---|
| `cat_extras_ped` | CatBoost（対称木 + ordered boosting、カテゴリの扱いが違う）。v2 には素の特徴量の CatBoost しか入っていなかった |
| `lgb_dart_extras_ped` | LightGBM dart（木のドロップアウト。過適合の出方が違う） |
| `lgb_et_extras_ped` | LightGBM extra_trees（分割点をランダム化。分散が小さい別の近似） |
| `xgb_deep_extras_ped` | XGBoost の浅い・遅い・列サンプリング強め設定（ハイパーパラメータ構成の多様化） |
| `nn_tabm_extras_ped` | TabM-mini（入力側アダプタ k=16 の暗黙アンサンブル NN。TabM 論文、実装は自前） |
| `nn_noise_extras_ped` | NN に学習時の入力ガウスノイズ（σ=0.1）を入れた版 |

各メンバーを、近い既存メンバーと LogLoss で比べ、保存した OOF は tp-05m（多層スタッキング）の候補プールに加わる。採否は最終的に「アンサンブルが v2 を有意に上回るか」で決める（tp-05m）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, win_models as wm, zoo, zoo_runner
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_n"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP05N_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1, 2]
mlflow.set_experiment("keiba-tp-model-zoo")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP, XP, NP = zoo.tuned_params("lgb", best["lgb"]["params"]), zoo.tuned_params("xgb", best["xgb"]["params"]), zoo.tuned_params("nn", best["nn"]["params"])
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]

In [ ]:
def lgb_variant(over, rounds_mult=1.0):
    return {"lgb": {**LP["lgb"], **over}, "n_rounds": int(LP["n_rounds"] * rounds_mult)}


MEMBERS = {
    "cat_extras_ped": ("cat_binary", {"task_type": "GPU"}, "cat_binary"),
    "lgb_dart_extras_ped": ("lgb_binary", lgb_variant({"boosting": "dart", "drop_rate": 0.1, "skip_drop": 0.5}), "lgb_extras_ped"),
    "lgb_et_extras_ped": ("lgb_binary", lgb_variant({"extra_trees": True, "num_leaves": 63, "feature_fraction": 0.5, "min_data_in_leaf": 80}, 1.5), "lgb_extras_ped"),
    "xgb_deep_extras_ped": ("xgb_binary", {"device": "cuda", "max_depth": 4, "learning_rate": 0.03, "n_estimators": 800, "subsample": 0.7, "colsample_bytree": 0.4, "reg_lambda": 10.0, "min_child_weight": 50}, "xgb_extras_ped"),
    "nn_tabm_extras_ped": ("nn", {**NP, "tabm_k": 16}, "nn_extras_ped"),
    "nn_noise_extras_ped": ("nn", {**NP, "noise": 0.1}, "nn_extras_ped"),
}
ROWS = []
for name, (recipe, params, ref) in MEMBERS.items():
    t0 = time.time()
    with mlflow.start_run(run_name=f"tp-05n-{name}", tags={"notebook": "tp-05n", "git_commit": GIT_COMMIT, "variant": name, "ref": ref}):
        res, oof = zoo_runner.run_oof(name, recipe, X, meta, folds, SEEDS, THREADS, params=dict(params), out_dir=OOF_DIR)
        mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in res.columns if c not in ("model", "fold")})
    b = pd.read_parquet(OOF_DIR / f"{ref}.parquet")
    m = b.merge(oof, on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    row = {"variant": name, "基準": ref}
    for t, k in (("win", 1), ("top2", 2), ("top3", 3)):
        yy = (fin <= k).astype(float)
        row[f"{t}_ref"], row[f"{t}_new"] = wm.binary_logloss(m[f"p_{t}_b"].to_numpy(), yy), wm.binary_logloss(m[f"p_{t}_v"].to_numpy(), yy)
        row[f"Δ{t}"] = row[f"{t}_new"] - row[f"{t}_ref"]
    # 基準との予測の相関（logit）: 低いほど多様
    z = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    row["logit相関(win)"] = float(np.corrcoef(z(m["p_win_b"].to_numpy()), z(m["p_win_v"].to_numpy()))[0, 1])
    row["sec"] = time.time() - t0
    ROWS.append(row)
    print(f"{name:22s} win {row['win_new']:.5f} (基準 {row['win_ref']:.5f}) top2 {row['top2_new']:.5f} top3 {row['top3_new']:.5f} 相関 {row['logit相関(win)']:.3f} ({row['sec']:.0f}s)", flush=True)
    gc.collect()
R = pd.DataFrame(ROWS).set_index("variant")
display(R.round(5)); R.to_csv(OUT / "members.csv")
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "members": R.round(6).reset_index().to_dict("records")}, ensure_ascii=False, indent=1, default=str))
with mlflow.start_run(run_name="tp-05n-summary", tags={"notebook": "tp-05n", "git_commit": GIT_COMMIT}):
    mlflow.log_artifact(str(OUT / "members.csv"), "improve_n")